# Simulated Annealing for Vehicle Routing

We build delivery routes that visit every customer while respecting vehicle capacity and service time windows. The search decides both **which vehicle serves each customer** and **the order of visits**.

The notebook follows four steps:

1. Load the instances and define an independent feasibility evaluator.
2. Construct a feasible initial solution and improve it with Simulated Annealing (SA).
3. Repeat two diagnostic experiments over ten seeds to observe variability.
4. Evaluate distance-only optimization on 24 held-out instances for comparison with Tabu Search.

Each experiment uses a **20-second search budget per run**, including construction. We report distance, fleet size, feasibility, elapsed time and a reference gap.


## Setup

Run the cells from top to bottom. Uncomment the installation line if dependencies are missing. `vrplib` reads the instance and solution files, NumPy stores the numerical data, pandas builds result tables, and Numba compiles the static search loops.

The setup prints the Python version, platform and package versions to help identify the execution environment. It locates the repository root so the notebook works when started from either the root or `notebooks/`. Downloaded files go to `data/solomon/`, which is git-ignored.

A seed fixes the random choices, but the wall-clock stopping rule can allow different numbers of moves on different machines. Repeated runs can therefore differ slightly even with the same seed.


In [1]:

# %pip install numpy pandas matplotlib vrplib numba
from pathlib import Path
import json, hashlib, urllib.request, platform, sys, importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import vrplib

print('Python:', sys.version.split()[0], '| Machine:', platform.platform())
print({name: importlib.metadata.version(name) for name in ['numpy','pandas','matplotlib','vrplib']})
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').exists())
DATA = ROOT / 'data' / 'solomon'
DATA.mkdir(parents=True, exist_ok=True)

if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))


Python: 3.14.7 | Machine: Windows-11-10.0.26300-SP0
{'numpy': '2.5.3', 'pandas': '3.0.6', 'matplotlib': '3.11.2', 'vrplib': '2.2.0'}


## Data

The next cell downloads missing `.vrp` instances and `.sol` reference routes from PyVRP/Instances. Existing files are reused. A SHA256 hash identifies the exact bytes used; it does not independently certify their source or correctness.

We separate the experiments by purpose:

| Experiment | Instances | Seeds | Purpose |
| --- | --- | --- | --- |
| Small diagnostic | First 25 customers of C101 | 0-9 | Check the pipeline on a smaller problem and measure seed variability |
| Full diagnostic | RC101, 100 customers | 0-9 | Observe behavior on a full-size instance |
| Held-out comparison | 24 full Solomon instances | 0 per instance | Cover six families using distance-only optimization |

The held-out set is C102-C105, C202-C205, R102-R105, R202-R205, RC102-RC105 and RC202-RC205. It excludes C101, C201, R101, R201, RC101 and RC201, which were used for parameter tuning. The two diagnostic instances are therefore not held-out evidence.

Reference routes are loaded for evaluation only. They are never passed to the search as starting solutions or used to choose moves.


In [ ]:

from pathlib import Path
import hashlib
import urllib.request
import vrplib

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').exists())
DATA = ROOT / 'data' / 'solomon'
DATA.mkdir(parents=True, exist_ok=True)

BASE_URL = 'https://raw.githubusercontent.com/PyVRP/Instances/main/VRPTW/'
FILES = {
    'C101.vrp': 'Solomon/C101.vrp',
    'C101.sol': 'Solomon/C101.sol',
    'RC101.vrp': 'Solomon/RC101.vrp',
    'RC101.sol': 'Solomon/RC101.sol',
}

HELD_OUT = [f'{family}{i}' for family in ('C', 'R', 'RC')
            for start in (102, 202) for i in range(start, start + 4)]
assert len(HELD_OUT) == 24 and len(set(HELD_OUT)) == 24
for name in HELD_OUT:
    for ext in ('vrp', 'sol'):
        FILES[f'{name}.{ext}'] = f'Solomon/{name}.{ext}'
for local_name, remote_path in FILES.items():
    target = DATA / local_name
    if not target.exists():
        urllib.request.urlretrieve(BASE_URL + remote_path, target)
    print(local_name, '| SHA256:', hashlib.sha256(target.read_bytes()).hexdigest())

raw_small = vrplib.read_instance(DATA / 'C101.vrp')
raw_rc = vrplib.read_instance(DATA / 'RC101.vrp')
reference_small = vrplib.read_solution(DATA / 'C101.sol')
reference_rc = vrplib.read_solution(DATA / 'RC101.sol')


C101.vrp | SHA256: 048312f0b95a6e474736f18e951322a002762016a08adb2d0cdf5688ad0a8192
C101.sol | SHA256: ee07c932bc4200e0782e25fccdf8f3b7548ce07b841f66906d6dce8973be0f00
RC101.vrp | SHA256: aa85524e59f3f52106bf6de262afdff81dbeeb769dba299cb14ead03e9955626
RC101.sol | SHA256: 866e8ecce2f6dfd9835a2af30322464dda9ce13446397c5eae70b28e687e1f11
C102.vrp | SHA256: f1a7c1473aa13fdef92193525d9394f8671ae7f943a10cad37e17bda0879e14f
C102.sol | SHA256: ee07c932bc4200e0782e25fccdf8f3b7548ce07b841f66906d6dce8973be0f00
C103.vrp | SHA256: 70471412625d1745bc680def436a21cdf67c44946c6d1f10544013d1d66a6676
C103.sol | SHA256: 675b45a47a6f1433bc9606349ca4baafb0ba255248ca20a68c6b3495d2dba17b
C104.vrp | SHA256: 58930b926823417d6989a1872ab73dbd2df8238a211e8432f1c2638a795b4dd9
C104.sol | SHA256: c9759507f943c984c4a2475966d1058afc42a2158bd75f5052461fc2fc552e15
C105.vrp | SHA256: 7d90b35d578b2b2a5bcc8ceaaf59500ca33890a434cedba48c72d39d01d2c08e
C105.sol | SHA256: ee07c932bc4200e0782e25fccdf8f3b7548ce07b841f66906d6dce8

## Solution representation and feasibility

Customer indices are `1, ..., n`; index `0` is the depot. A solution such as `[[1, 4], [2, 3]]` means two routes: `0 -> 1 -> 4 -> 0` and `0 -> 2 -> 3 -> 0`. The depot is implicit at both ends.

`make_problem` extracts coordinates, demands, service durations, time windows and the fleet limit. It builds Euclidean distances, optionally truncating each arc to one decimal. The same selected distances are used as travel times in static modes.

For each route, `simulate` starts at the depot's opening time, travels to the next customer, waits if it arrives before the opening time, and adds the service duration. Service must start before the customer's closing time. The vehicle must also return before the depot closes, and its total demand must fit its capacity.

`validate` checks every customer appears exactly once, the fleet limit is respected, and every route is feasible. It then recomputes distance and schedules from scratch. This independent check is also applied to the final routes returned by the compiled search.

Traffic mode is a separate approximation: travel multipliers depend on departure time, and depot departures are searched on a discrete grid. The experiments below use static travel times and do not validate the traffic model.


In [3]:
import math
import time
from dataclasses import dataclass
import numpy as np

@dataclass
class Problem:
    name: str
    xy: np.ndarray
    q: np.ndarray
    tw: np.ndarray
    service: np.ndarray
    capacity: float
    max_vehicles: int
    distance: np.ndarray
    mode: str = 'benchmark'
    departure_step: float = 15.0
    distance_convention: str = 'truncated'

    @property
    def n(self):
        return len(self.q) - 1


def make_problem(raw, n=None, mode='benchmark', distance_convention=None):
    if mode not in {'benchmark', 'distance', 'traffic'}:
        raise ValueError('Unknown optimization mode')
    distance_convention = distance_convention or ('truncated' if mode == 'benchmark' else 'exact')
    if distance_convention not in {'truncated', 'exact'}:
        raise ValueError('Unknown distance convention')
    n = len(raw['demand']) - 1 if n is None else n
    xy = np.array(raw['node_coord'][:n + 1], dtype=float)
    d = np.linalg.norm(xy[:, None] - xy[None, :], axis=2)
    # Convention DIMACS/PyVRP: truncate each arc to one decimal.
    if distance_convention == 'truncated':
        d = np.floor(d * 10 + 1e-9) / 10
    service = np.asarray(raw['service_time'], dtype=float)
    if service.ndim == 0:
        service = np.full(n + 1, float(service))
    else:
        service = service[:n + 1].copy()
    service[0] = 0
    return Problem(raw.get('name', 'instance') + f'-{n}', xy,
                   np.array(raw['demand'][:n + 1], dtype=float),
                   np.array(raw['time_window'][:n + 1], dtype=float),
                   service, float(raw['capacity']), int(raw.get('vehicles', n)), d, mode, distance_convention=distance_convention)

RHO = np.array([1, 1, 1, 1, 1, 1.05, 1.30, 1.75, 1.80, 1.40, 1.20, 1.25,
                1.30, 1.25, 1.20, 1.25, 1.45, 1.80, 1.75, 1.40, 1.10, 1.05, 1, 1])


def simulate(p, route, departure=0.0):
    """Independent forward schedule; None if infeasible. Times use instance units."""
    if not route:
        return dict(distance=0., duration=0., departure=departure, return_time=departure, rows=[])
    if sum(p.q[i] for i in route) > p.capacity + 1e-8:
        return None
    if departure < p.tw[0, 0] - 1e-8 or departure > p.tw[0, 1] + 1e-8:
        return None
    t, prev, distance, rows = departure, 0, 0., []
    for j in list(route) + [0]:
        length = p.distance[prev, j]
        multiplier = RHO[int(t // 60) % 24] if p.mode == 'traffic' else 1.
        arrive = t + length * multiplier
        start = max(arrive, p.tw[j, 0])
        if start > p.tw[j, 1] + 1e-8:
            return None
        rows.append(dict(customer=j, leave_previous=t, arrival=arrive,
                         start=start, waiting=start-arrive))
        t = start + p.service[j]
        distance += length
        prev = j
    return dict(distance=distance, duration=t-departure, departure=departure,
                return_time=t, rows=rows)


def evaluate_route(p, route):
    if not route:
        return 0., None
    if p.mode != 'traffic':
        sch = simulate(p, route, p.tw[0, 0])
    else:
        # Approximation explicit: grid of depot departures, earliest service thereafter.
        sch = None
        for dep in np.arange(p.tw[0, 0], p.tw[0, 1] + 1e-8, p.departure_step):
            candidate = simulate(p, route, float(dep))
            if candidate is not None and (sch is None or candidate['duration'] < sch['duration']):
                sch = candidate
    if sch is None:
        return math.inf, None
    return sch['distance' if p.mode != 'traffic' else 'duration'], sch


def validate(p, routes):
    """Recompute from scratch: coverage, capacity, windows, fleet, returned objective."""
    flat = [i for r in routes for i in r]
    assert sorted(flat) == list(range(1, p.n+1)), 'Missing/duplicate/out-of-range customer'
    routes = [r for r in routes if r]
    assert len(routes) <= p.max_vehicles, 'Fleet limit exceeded'
    schedules = []
    for r in routes:
        assert sum(p.q[i] for i in r) <= p.capacity + 1e-8
        cost, sch = evaluate_route(p, r)
        assert math.isfinite(cost), 'Time window/depot violation'
        schedules.append(sch)
    return dict(vehicles=len(routes), distance=sum(s['distance'] for s in schedules),
                duration=sum(s['duration'] for s in schedules), schedules=schedules)



## Why Simulated Annealing?

A move changes customer assignments or visit order. Accepting only improvements can trap the search in a local optimum: every available small change looks worse, even though several changes together could improve the solution.

SA also accepts some worsening moves. Let $E$ be the selected objective and $\Delta = E_{new} - E_{current}$:

- If $\Delta \leq 0$, accept the move.
- Otherwise, accept it with probability $\exp(-\Delta/T)$, where $T$ is the temperature.

Higher temperatures allow more exploration; lower temperatures make acceptance more selective. Infeasible candidates are rejected regardless of temperature. The algorithm keeps a separate **best feasible solution**, so accepting a worse current solution never loses the best result already found.

SA explores only part of the search space. It does not prove optimality or guarantee a particular gap on other instances or hardware.


## Initial solution and neighborhood moves

`initialize` builds a solution without reference information. `repair` shuffles customers and inserts each into a feasible position. It considers up to eight nearby existing routes, usually chooses the cheapest insertion, and sometimes selects another good insertion to diversify construction. A new route is opened when no considered insertion is feasible and the fleet limit permits it. Failed construction can be retried within the time budget.

The compiled static search then samples five move types:

| Move | What changes |
| --- | --- |
| Relocate | Move one customer to another route |
| Swap | Exchange two customers, within one route or between routes |
| Segment reversal | Reverse a sequence within a route (2-opt) |
| Short-chain relocation | Move a consecutive block of two or three customers to another route |
| Route-tail exchange | Exchange the ends of two routes (2-opt-star), which can also merge routes |

Empty routes do not count as vehicles. Every proposed move must preserve capacity and time-window feasibility before the SA acceptance rule is applied.

The Python fallback instead combines destroy-and-repair, swaps and reversals. The `repair` function below is used for initialization by the compiled solver as well as for moves by that fallback.


In [4]:
def repair(p, routes, removed, rng, route_limit=8, deadline=math.inf):
    """Randomized cheapest feasible insertion; a new route is permitted."""
    routes = [list(r) for r in routes if r]
    rng.shuffle(removed)
    for customer in removed:
        if time.perf_counter() >= deadline:
            raise TimeoutError("SA time budget exhausted")
        options = []
        # Limit candidate routes by proximity for large instances.
        candidates = sorted(range(len(routes)), key=lambda k:
                            min(p.distance[customer, j] for j in routes[k]))[:route_limit]
        for k in candidates:
            r = routes[k]
            if sum(p.q[i] for i in r) + p.q[customer] > p.capacity + 1e-8:
                continue
            old, _ = evaluate_route(p, r)
            for pos in range(len(r)+1):
                if time.perf_counter() >= deadline:
                    raise TimeoutError("SA time budget exhausted")
                nr = r[:pos] + [customer] + r[pos:]
                cost, _ = evaluate_route(p, nr)
                if math.isfinite(cost):
                    options.append((cost-old, k, nr))
        if options:
            options.sort(key=lambda x: x[0])
            # Usually greedy, sometimes diversify between best insertions.
            choice = 0 if rng.random() < .9 else int(rng.integers(min(3, len(options))))
            _, k, nr = options[choice]
            routes[k] = nr
        elif len(routes) < p.max_vehicles and math.isfinite(evaluate_route(p, [customer])[0]):
            routes.append([customer])
        else:
            return None
    return routes


def initialize(p, rng, deadline=math.inf):
    # Independent of all reference routes and values.
    for _ in range(50):
        out = repair(p, [], list(range(1, p.n+1)), rng, deadline=deadline)
        if out is not None:
            return out
    raise RuntimeError('No feasible initialization; increase construction effort or check data.')



## Objective, cooling and time budget

The objective and the distance convention are separate settings:

| Mode | Objective | Default distance convention |
| --- | --- | --- |
| `benchmark` | Minimize vehicles first, then distance | Each arc truncated to one decimal |
| `distance` | Minimize total distance alone | Exact Euclidean distances |
| `traffic` | Minimize approximate total route duration | Exact Euclidean distances |

Benchmark energy is $E = M K + D$, where $K$ is the number of nonempty routes, $D$ is distance and $M = 2n\max_{ij}(d_{ij}) + 1$. This weight is larger than any possible total route distance, so one fewer vehicle takes priority. In distance mode, $E = D$: adding a vehicle is allowed if it reduces distance and respects the fleet limit.

Static experiments call `anneal(..., accelerated=True)`, implemented in `src/cvrp/sa_static.py`. Their cooling follows elapsed time:

$$T(t) = T_0\left(\frac{0.02}{T_0}\right)^{t/L}, \qquad T_0 = \max(1,\;0.005D_{initial}),\quad L=20\text{ s}.$$

This cools from 0.5% of initial distance (with a minimum of 1) toward 0.02 across the budget. It avoids the almost constant temperature produced by slow iteration-based cooling in a short run. `ALPHA` and `T_MIN` apply only to the Python fallback; they do not control compiled static runs.

Numba is warmed up once on a synthetic instance before the experiments. Compilation is excluded as setup. The timer then includes initialization and search; `elapsed_s` also includes final independent validation. The deadline is checked between small batches of moves, so batch completion and validation can add a small measured overhead.

Reference costs are used only for reporting. They do not influence initialization, acceptance or stopping.


In [5]:
from cvrp.sa_static import anneal_static, warmup

# Synthetic compilation only: no benchmark data or reference routes are used.
warmup()

def anneal(p, seed=0, alpha=0.995, min_temperature=0.001, initial=None, time_limit=20.0, accelerated=False):
    if not 0 < alpha < 1:
        raise ValueError('alpha must be strictly between 0 and 1')
    if min_temperature <= 0:
        raise ValueError('min_temperature must be positive')
    if not math.isfinite(time_limit) or time_limit <= 0:
        raise ValueError('time_limit must be finite and positive')
    if accelerated and p.mode != 'traffic':
        return anneal_static(p, initialize, validate, seed=seed, initial=initial, time_limit=time_limit)
    rng = np.random.default_rng(seed)
    started = time.perf_counter()
    deadline = started + time_limit
    current = initialize(p, rng, deadline) if initial is None else [list(r) for r in initial]
    validate(p, current)
    # Safe distance upper bound for n customers and <=n nonempty routes.
    # Thus one fewer vehicle always dominates any distance change in benchmark mode.
    vehicle_weight = 2 * p.n * float(p.distance.max()) + 1
    cache = {}
    def route_cost(r):
        key = tuple(r)
        if key not in cache:
            cache[key] = evaluate_route(p, r)[0]
        return cache[key]
    def energy(rs):
        return sum(route_cost(r) for r in rs) + (vehicle_weight*len(rs) if p.mode == 'benchmark' else 0)
    current_e = energy(current)
    best, best_e = [r[:] for r in current], current_e
    initial_metrics = validate(p, current)
    # Temperature controls distance worsening; vehicle reduction remains strongly preferred.
    initial_temperature = max(1., .05 * sum(route_cost(r) for r in current))
    temperature = initial_temperature
    history = [(0., sum(route_cost(r) for r in best), len(best))]
    iteration = accepted = 0
    while temperature > min_temperature and time.perf_counter() < deadline:
        iteration += 1
        candidate = [r[:] for r in current]
        if rng.random() < .65:
            # Ruin/repair changes assignment and can eliminate a complete route.
            k = int(rng.integers(len(candidate)))
            if rng.random() < .35:
                removed = candidate.pop(k)
            else:
                pool = [i for r in candidate for i in r]
                count = min(len(pool), int(rng.integers(2, min(12, p.n)+1)))
                removed = rng.choice(pool, count, replace=False).tolist()
                delete = set(removed)
                candidate = [[i for i in r if i not in delete] for r in candidate]
            try:
                candidate = repair(p, candidate, removed, rng, deadline=deadline)
            except TimeoutError:
                break
        else:
            a, b = rng.integers(len(candidate), size=2)
            if a == b:
                if len(candidate[a]) < 2:
                    continue
                u, v = sorted(rng.choice(len(candidate[a]), 2, replace=False))
                candidate[a][u:v+1] = reversed(candidate[a][u:v+1])
            else:
                u, v = int(rng.integers(len(candidate[a]))), int(rng.integers(len(candidate[b])))
                candidate[a][u], candidate[b][v] = candidate[b][v], candidate[a][u]
        if candidate is None:
            continue
        candidate = [r for r in candidate if r]
        new_e = energy(candidate)
        delta = new_e-current_e
        if math.isfinite(new_e) and (delta <= 0 or rng.random() < math.exp(-delta/temperature)):
            current, current_e = candidate, new_e
            accepted += 1
            if new_e < best_e - 1e-8:
                best, best_e = [r[:] for r in current], new_e
                history.append((time.perf_counter()-started, sum(route_cost(r) for r in best), len(best)))
        if len(cache) > 100000:
            cache.clear()
        temperature *= alpha
    search_elapsed = time.perf_counter()-started
    metrics = validate(p, best)
    return dict(routes=best, metrics=metrics, initial=initial_metrics, seed=seed,
                elapsed=time.perf_counter()-started, search_elapsed=search_elapsed,
                time_limit=time_limit, stop_reason=('time_limit' if time.perf_counter() >= deadline else 'temperature'),
                iterations=iteration,
                accepted=accepted, history=history)


## Reference checks

Before reporting a gap, we validate reference routes using the same evaluator and recompute their distance under the chosen convention. The diagnostic checks below also verify the expected truncated-distance values. C101-25 uses the C101 reference routes restricted to customers 1-25, with empty routes removed; this is a constructed comparison route set, not a certified optimum for the reduced instance.

A route feasible with truncated travel times can violate a time window when arcs use exact Euclidean distances. Distance-only experiments mark such references with `reference_feasible=False` and leave their gap unavailable (`NaN`). The SA solution is still independently validated and its distance, fleet size and runtime remain available.

The supplied `.sol` routes are fleet-first references. Their distances are useful comparison points, but they are not certified optima for the distance-only objective. A fair SA-versus-Tabu comparison must use the same customers, objective, distance and travel-time conventions, constraints, seeds and time budget.


In [6]:
p25 = make_problem(raw_small, 25)
pRC = make_problem(raw_rc)
ref25 = [[int(i) for i in route if i <= 25] for route in reference_small['routes']]
ref25 = [r for r in ref25 if r]
refRC = [list(map(int, r)) for r in reference_rc['routes']]

assert np.isclose(validate(p25, ref25)['distance'], 191.3)
assert np.isclose(validate(pRC, refRC)['distance'], reference_rc['cost'])
for p, routes in [(p25, ref25), (pRC, refRC)]:
    result = validate(p, routes)
    print(f'{p.name}: {p.n} customers, {result["vehicles"]} reference routes, '
          f'distance {result["distance"]:.1f} - feasible')


C101-25: 25 customers, 3 reference routes, distance 191.3 - feasible
RC101-100: 100 customers, 15 reference routes, distance 1619.8 - feasible


## Reading the gap and result columns

For a feasible reference, we compute

$$\mathrm{gap\_pct}=100\left(\frac{D_{SA}}{D_{reference}}-1\right).$$

For example, distance 1,050 against a reference distance of 1,000 gives a 5% gap. A negative gap means SA found a shorter distance than the supplied reference; it does not prove global optimality. A missing gap means the reference failed validation under the selected convention, not that SA achieved zero gap.

| Column | Interpretation |
| --- | --- |
| `seed` | Random seed for one run |
| `vehicles`, `distance` | Nonempty route count and total travel distance |
| `reference_distance`, `reference_vehicles` | Recomputed distance and fleet size of the comparison routes |
| `reference_feasible` | Whether those routes passed validation |
| `gap_pct` | Measured relative difference from the reference distance |
| `elapsed_s`, `iterations` | Measured total execution time and proposed moves |
| `initial_distance`, `initial_vehicles` | Construction quality before SA starts |
| `objective`, `distance_convention`, `engine` | Settings needed to interpret and reproduce the result |

The mean summarizes the runs; the minimum and maximum show their spread. The observation cells calculate these statistics directly from the result tables so the text follows the executed output.


## Small-instance experiment: C101-25

We run seeds 0-9 independently, reconstructing an initial solution for each seed. Benchmark mode uses truncated distances and gives fleet size priority. Each compiled run has the same 20-second budget.

`run_experiment` validates the reference, runs SA, computes the gap and builds one row per seed.

The printed table shows each run rather than only the best seed. The summary reports mean and range, and the final line displays the routes of the best benchmark solution selected by fleet size and then distance. The later observation cell describes the smallest distance in the table, so its criterion is explicitly distance.


In [7]:
def print_gap_conclusions(name, table):
    best = table.loc[table.distance.idxmin()]
    print(f"\nConclusion - {name}: mean gap {table.gap_pct.mean():.2f}%; "
          f"range {table.gap_pct.min():.2f}% to {table.gap_pct.max():.2f}%.")
    print(f"Best distance: seed {int(best.seed)}, {best.gap_pct:.2f}%, "
          f"{int(best.vehicles)} trucks vs {int(best.reference_vehicles)} in the reference.")
    if len(table) == 1:
        print('Only one seed: variability is not yet measured.')

def run_experiment(p, ref_routes, seeds, alpha=0.995, min_temperature=0.001, time_limit=20.0):
    try:
        ref = validate(p, ref_routes)
        reference_feasible = True
    except AssertionError:
        if p.mode != 'distance':
            raise
        # Truncated-distance reference routes can violate exact travel-time windows.
        assert sorted(i for r in ref_routes for i in r) == list(range(1, p.n + 1))
        ref = dict(vehicles=len(ref_routes), distance=sum(
            p.distance[a, b] for r in ref_routes
            for a, b in zip([0] + r, r + [0])))
        reference_feasible = False
    results, records = [], []
    for seed in seeds:
        out = anneal(p, seed=seed, alpha=alpha, min_temperature=min_temperature, time_limit=time_limit, accelerated=True)
        m = out['metrics']
        gap = 100 * (m['distance'] / ref['distance'] - 1) if reference_feasible else np.nan
        records.append(dict(instance=p.name, n=p.n,
                            algorithm=f'sa_{p.mode}_{p.distance_convention}',
                            cost=m['distance'], seconds=out['elapsed'], gap=gap,
                            feasible=True, reference_feasible=reference_feasible, objective=p.mode,
                            engine='compiled_static' if p.mode != 'traffic' else 'python_traffic',
                            distance_convention=p.distance_convention, time_limit_s=time_limit,
                            stop_reason=out['stop_reason'], seed=seed, vehicles=m['vehicles'], distance=m['distance'],
                            gap_pct=gap, reference_distance=ref['distance'],
                            reference_vehicles=ref['vehicles'],
                            elapsed_s=out['elapsed'], iterations=out['iterations'],
                            initial_distance=out['initial']['distance'],
                            initial_vehicles=out['initial']['vehicles']))
        results.append(out)
    return pd.DataFrame(records), results

SEEDS = list(range(10))
ALPHA = 0.995
T_MIN = 0.001
results25, runs25 = run_experiment(p25, ref25, SEEDS, alpha=ALPHA, min_temperature=T_MIN)
print(results25[['seed','vehicles','distance','reference_distance','gap_pct','elapsed_s']].round(2).to_string(index=False))
print_gap_conclusions(p25.name, results25)
best25 = min(runs25, key=lambda o: (o['metrics']['vehicles'], o['metrics']['distance']))
print('Best routes:', best25['routes'])


 seed  vehicles  distance  reference_distance  gap_pct  elapsed_s
    0         3     191.3               191.3      0.0       20.0
    1         3     191.3               191.3      0.0       20.0
    2         3     191.3               191.3      0.0       20.0
    3         3     191.3               191.3      0.0       20.0
    4         3     191.3               191.3      0.0       20.0
    5         3     191.3               191.3      0.0       20.0
    6         3     191.3               191.3      0.0       20.0
    7         3     191.3               191.3      0.0       20.0
    8         3     191.3               191.3      0.0       20.0
    9         3     191.3               191.3      0.0       20.0

Conclusion - C101-25: mean gap 0.00%; range 0.00% to 0.00%.
Best distance: seed 0, 0.00%, 3 trucks vs 3 in the reference.
Best routes: [[5, 3, 7, 8, 10, 11, 9, 6, 4, 2, 1], [13, 17, 18, 19, 15, 16, 14, 12], [20, 24, 25, 23, 22, 21]]


## Full-instance diagnostic: RC101

We repeat the experiment with all 100 RC101 customers and seeds 0-9, again using benchmark mode and a 20-second budget per run. This checks whether the search remains effective at full size and shows how sensitive the result is to random choices.

Compare distance and truck count together because the objective gives vehicles priority. The per-seed elapsed times verify the budget actually used. The `ALPHA` and `T_MIN` variables in the next cell are retained for the fallback implementation; the compiled static solver uses the wall-clock schedule described above.

The next two observation cells generate their text from the diagnostic result tables. Their figures therefore update when the experiments are rerun.


In [8]:
ALPHA = 0.999
T_MIN = 0.00001
N_SEEDS_RC101 = 10

rc_results, rc_runs = run_experiment(
    pRC, refRC, range(N_SEEDS_RC101), alpha=ALPHA, min_temperature=T_MIN
)
print(rc_results[['seed','vehicles','distance','reference_distance','gap_pct','elapsed_s']]
      .round(2).to_string(index=False))
print_gap_conclusions(pRC.name, rc_results)


 seed  vehicles  distance  reference_distance  gap_pct  elapsed_s
    0        15    1640.9              1619.8     1.30       20.0
    1        15    1631.3              1619.8     0.71       20.0
    2        15    1633.8              1619.8     0.86       20.0
    3        15    1641.0              1619.8     1.31       20.0
    4        15    1643.4              1619.8     1.46       20.0
    5        16    1655.1              1619.8     2.18       20.0
    6        15    1639.5              1619.8     1.22       20.0
    7        15    1631.3              1619.8     0.71       20.0
    8        15    1634.6              1619.8     0.91       20.0
    9        15    1631.3              1619.8     0.71       20.0

Conclusion - RC101-100: mean gap 1.14%; range 0.71% to 2.18%.
Best distance: seed 1, 0.71%, 15 trucks vs 15 in the reference.


In [9]:
from IPython.display import Markdown, display
table = results25
best = table.loc[table.distance.idxmin()]
display(Markdown(f'### Observations - C101-25\n\n'
    f'Across {len(table)} runs, the mean distance gap is {table.gap_pct.mean():.2f}%, '
    f'ranging from {table.gap_pct.min():.2f}% to {table.gap_pct.max():.2f}%. '
    f'The best distance uses seed {int(best.seed)}, {int(best.vehicles)} trucks, '
    f'and distance {best.distance:,.1f} (reference {best.reference_distance:,.1f}, '
    f'{int(best.reference_vehicles)} trucks). '
    f'Mean elapsed time: {table.elapsed_s.mean():.3f} s; budget: 20 s per run.'))


### Observations - C101-25

Across 10 runs, the mean distance gap is 0.00%, ranging from 0.00% to 0.00%. The best distance uses seed 0, 3 trucks, and distance 191.3 (reference 191.3, 3 trucks). Mean elapsed time: 20.000 s; budget: 20 s per run.

In [10]:
from IPython.display import Markdown, display
table = rc_results
best = table.loc[table.distance.idxmin()]
display(Markdown(f'### Observations - RC101-100\n\n'
    f'Across {len(table)} runs, the mean distance gap is {table.gap_pct.mean():.2f}%, '
    f'ranging from {table.gap_pct.min():.2f}% to {table.gap_pct.max():.2f}%. '
    f'The best distance uses seed {int(best.seed)}, {int(best.vehicles)} trucks, '
    f'and distance {best.distance:,.1f} (reference {best.reference_distance:,.1f}, '
    f'{int(best.reference_vehicles)} trucks). '
    f'Mean elapsed time: {table.elapsed_s.mean():.3f} s; budget: 20 s per run.'))


### Observations - RC101-100

Across 10 runs, the mean distance gap is 1.14%, ranging from 0.71% to 2.18%. The best distance uses seed 1, 15 trucks, and distance 1,631.3 (reference 1,619.8, 15 trucks). Mean elapsed time: 20.000 s; budget: 20 s per run.

## Held-out distance-only comparison

We now switch to **distance-only optimization with exact Euclidean distances**, matching the project's distance objective. Each of the 24 held-out instances is run once with seed 0 and a 20-second budget. Vehicle count remains a constraint and a reported metric, but it carries no penalty in the objective.

For each instance, the next cell loads the data, builds the problem, checks the reference and runs SA. It collects all rows in the in-memory `comparison` table, along with the detailed settings and feasibility fields.

The grouped table reports mean gap and elapsed time by family. pandas excludes missing gaps from those means. Reference feasibility is reported separately, and all runs remain in the comparison table.




In [11]:
comparison_tables = []
for name in HELD_OUT:
    raw = vrplib.read_instance(DATA / f'{name}.vrp')
    reference = vrplib.read_solution(DATA / f'{name}.sol')
    problem = make_problem(raw, mode='distance', distance_convention='exact')
    routes = [list(map(int, r)) for r in reference['routes']]
    table, _ = run_experiment(problem, routes, seeds=[0], alpha=0.999999,
                              min_temperature=1e-5, time_limit=20.0)
    comparison_tables.append(table)
    print(table[['instance', 'seed', 'vehicles', 'distance', 'gap_pct', 'elapsed_s',
                 'stop_reason', 'reference_feasible']].round(3).to_string(index=False), flush=True)
comparison = pd.concat(comparison_tables, ignore_index=True)
assert comparison.instance.nunique() >= 20
display(comparison.groupby(comparison.instance.str.extract(r"^(C|RC|R)([12])", expand=True)
                           .agg("".join, axis=1))[['gap_pct', 'elapsed_s']].mean())



instance  seed  vehicles  distance  gap_pct  elapsed_s stop_reason  reference_feasible
C102-100     0        10   828.937      0.0       20.0  time_limit                True


instance  seed  vehicles  distance  gap_pct  elapsed_s stop_reason  reference_feasible
C103-100     0        10   829.867    0.218     20.001  time_limit                True


instance  seed  vehicles  distance  gap_pct  elapsed_s stop_reason  reference_feasible
C104-100     0        10   824.777      0.0       20.0  time_limit                True


instance  seed  vehicles  distance  gap_pct  elapsed_s stop_reason  reference_feasible
C105-100     0        10   828.937      0.0       20.0  time_limit                True


instance  seed  vehicles  distance  gap_pct  elapsed_s stop_reason  reference_feasible
C202-100     0         3   591.557      0.0       20.0  time_limit                True


instance  seed  vehicles  distance  gap_pct  elapsed_s stop_reason  reference_feasible
C203-100     0         4   605.207    2.374       20.0  time_limit                True


... (15 outputs left out)

 instance  seed  vehicles  distance  gap_pct  elapsed_s stop_reason  reference_feasible
RC203-100     0         6   950.398    2.535       20.0  time_limit                True


 instance  seed  vehicles  distance  gap_pct  elapsed_s stop_reason  reference_feasible
RC204-100     0         5   806.778    2.573       20.0  time_limit                True


 instance  seed  vehicles  distance  gap_pct  elapsed_s stop_reason  reference_feasible
RC205-100     0         8  1178.011    1.758       20.0  time_limit                True


,gap_pct,elapsed_s
C1,0.054403,20.000338
C2,0.859766,20.000282
R1,1.327322,20.000290
R2,0.799273,20.000269
RC1,1.601662,20.000262
RC2,2.220679,20.000267
